# 14.7 為什麼位置公式算得出，長文章卻未必讀得好？


把「校慶週六上午開始請在操場入口集合」依序寫成十六張卡片，每張卡放一個字。這裡為了只看位置，約定一張卡就是一個token；真實文字如何切成token，仍由所用的切詞工具決定。卡片從0編到15。假設一個模型訓練時，每段文字最多八張卡，它收到的位置便只有0到7。現在我們希望它連後面的八張也一起讀，位置公式算得出8到15，是否就表示它能讀好整段？

位置公式做的是替卡片提供位置資訊。回想用旋轉表示位置的RoPE：Query是某位置用來查找關聯的向量，Key是其他位置供它比對的向量；把其中成對的數字依位置旋轉，兩者的點積便會包含相對位置的影響。旋轉可以繼續算下去，轉到第15格不需要多查一張位置表。但這只是算出一組數字，模型還要學會在這組數字下，怎樣利用文字內容找出關聯。

在我們的八格假設裡，兩張卡最遠相隔七格。例如位置0和7相隔七格；位置0和15卻相隔十五格，這種距離沒有出現在原來的訓練段落中。熟悉位置之間也有很多不同詞句與關係，並不是在訓練範圍內就全部會答。把文章拉長，則又增加了原先根本遇不到的距離與位置組合。

下圖藍色卡片是原來可能出現的位置，橘色是延長後才會出現的位置。兩列只是把同一段文字折成兩行，第二列沒有重新從0編號。圖底把0到7、0到15的距離並排，請比較新增的是哪些位置關係。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-14-7-position-range.svg")))

這類把計算用到訓練範圍以外的做法，叫長度外推。它可能有用，但相對位置的公式本身沒有承諾外推後的品質。模型在熟悉範圍學到的比對方式，遇到新的旋轉組合時可能不再合適；後續還有把比對分數轉成注意力權重、混合內容和產生答案等步驟。因此，能算出長序列的向量，與能從長序列取出答案，中間仍有學習與驗收的工作。

這裡要分開三件事。訓練長度描述模型當時看過多長的片段；執行上限描述軟體這次允許送入多長的序列；任務能力則要看它能否完成問題。例如軟體接受十六張卡，表示輸入沒有因上限被丟掉；若要說它能取回遠處線索，還要提供答案明確的題目，核對完整回答。僅將上限從八改為十六，不會同時完成這三件事。

所以延長窗口時，先問位置規則怎麼處理新增範圍，再問模型如何適應，最後才問它在指定任務上有沒有讀好。下一節的做法是保留十六張卡，將位置刻度縮回原來八格所配置的座標範圍。

練習把假設改成「訓練每段最多十二張卡」，仍讓輸入有十六張。先指出哪些位置超過原段落的範圍，再算最遠的舊距離與新距離。原位置是0到11，新增位置是12到15；原來最遠相隔十一格，現在仍可相隔十五格。

<details>
<summary>回讀與權威來源</summary>

旋轉與相對點積可回讀[14.1](https://birdhackor.github.io/tiny-perceptron-vlm/14.1.html)。原始推導見[RoFormer，arXiv:2104.09864v5，§3.2及式14–16](https://arxiv.org/pdf/2104.09864v5)。直接外推的限制見[Position Interpolation，arXiv:2306.15595v2，§2.2](https://arxiv.org/pdf/2306.15595v2)；「允許長度」與任務上的有效長度之別，可讀[RULER，arXiv:2404.06654v3，§4](https://arxiv.org/pdf/2404.06654v3)。更多查證在[來源筆記](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-revision-20261005/sources/context.md)。本節十六張卡是紙上例子，沒有訓練或測試模型。

</details>
